# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [2]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

../../../../../../assets/03-wine-quality/inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [39]:
# TODO: implementa la lectura del CSV.

rows = []
row = {}

with open(DATA_PATH) as f:
    reader = csv.DictReader(f)

    flag = 1
    for r in reader:
        if flag:
            row = r
            flag = 0
        rows.append(r)

print("First row:", row)
print("All of them:")
for elem in rows:
    print(elem)
print(f"Number of rows: {len(rows)}")
columns = list(row.keys())
print(f"Columns: {columns}")

First row: {'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}
All of them:
{'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}
{'sample_id': 'red-002', 'fixed_acidity': '7.8', 'volatile_acidity': '0.88', 'citric_acid': '0.0', 'residual_sugar': '2.6', 'chlorides': '0.098', 'free_sulfur_dioxide': '25.0', 'total_sulfur_dioxide': '67.0', 'density': '0.9968', 'ph': '3.2', 'sulphates': '0.68', 'alcohol': '9.8'}
{'sample_id': 'red-003', 'fixed_acidity': '11.2', 'volatile_acidity': '0.28', 'citric_acid': '0.56', 'residual_sugar': '1.9', 'chl

### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?
- `sample_id`
2. ¿Cuántas características numéricas hay?
- 11
3. ¿Qué columna extra añadirías para probar el contrato?
- A pesar de que no se utilice en el modelo, añadiría también `sample_id` al contrato. De esta forma, podemos probar que el número no es inválido (lo que podría delatar otros errores).
4. ¿Qué valor de `ph` sería claramente inválido?
- Todos los que aparecen son válidos. Uno inválido sería uno que se diferenciase mucho de 3, por ejemplo, 20

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [40]:
class WineQualityRequestPydantic(BaseModel):
	model_config = ConfigDict(extra="forbid")

	fixed_acidity:			float = Field(gt=0, le=20)
	volatile_acidity:		float = Field(ge=0, le=2)
	citric_acid:			float = Field(ge=0, le=2)
	residual_sugar:			float = Field(gt=0, le=20)
	chlorides:				float = Field(ge=0, le=1)
	free_sulfur_dioxide:	float = Field(gt=0, le=100)
	total_sulfur_dioxide:	float = Field(gt=0, le=300)
	density:				float = Field(ge=0.98, le=1.01)
	ph:						float = Field(gt=2.5, le=4.5)
	sulphates:				float = Field(ge=0, le=3)
	alcohol:				float = Field(gt=5, le=20)


# TODO: separa sample_id de la fila y valida solo las características del modelo.
copy_row = row.copy()
copy_row.pop("sample_id")
request = WineQualityRequestPydantic.model_validate(copy_row)

### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [44]:
# TODO: completa los casos y captura ValidationError.

# 1: Valid
try:
    copy_row = row.copy()
    copy_row.pop("sample_id")
    request = WineQualityRequestPydantic.model_validate(copy_row)
except ValidationError as error:
    print(error)

# 2: Unexpected field
try:
    copy_row = row.copy()
    copy_row.pop("sample_id")
    copy_row["unexpected_field"] = 123
    request = WineQualityRequestPydantic.model_validate(copy_row)
except ValidationError as error:
    print(error)

# 3: Unexpected field
try:
    copy_row = row.copy()
    copy_row.pop("sample_id")
    copy_row["ph"] = 123
    request = WineQualityRequestPydantic.model_validate(copy_row)
except ValidationError as error:
    print(error)



1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/extra_forbidden
1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [ ]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
	fixed_acidity:			float
	volatile_acidity:		float
	citric_acid:			float
	residual_sugar:			float
	chlorides:				float
	free_sulfur_dioxide:	float
	total_sulfur_dioxide:	float
	density:				float
	ph:						float
	sulphates:				float
	alcohol:				float

copy_row = row.copy()
copy_row.pop("sample_id")
copy_row["ph"] = 99
request = WineQualityRequestDataclass(**copy_row)



### Decisión de diseño

Completa la frase:

> Usaría `Pydantic` en aquellas clases en las que necesite especificar rangos de valores permitidos para los atributos. Sin embargo, usaría `dataclass` en las clases en las que no fuese necesario, y quisiese simplemente controlar valores por defecto, post-inicializaciones...

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [53]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
	"fixed_acidity",
	"volatile_acidity",
	"citric_acid",
	"residual_sugar",
	"chlorides",
	"free_sulfur_dioxide",
	"total_sulfur_dioxide",
	"density",
	"ph",
	"sulphates",
	"alcohol",
)

# TODO: construye vector usando getattr(request, name).
vector = [[]]
for name in FEATURE_NAMES:
    vector[0].append(getattr(request, name))

print(vector)

[['7.4', '0.7', '0.0', '1.9', '0.076', '11.0', '34.0', '0.9978', 99, '0.56', '9.4']]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
    - Sí
- ¿El clasificador detectaría necesariamente el error?
    - No tiene por qué. El clasificador solo vería dos números y los procesaria, aunque el resultados posiblemente fuese incorrecto.
- ¿Cómo evitarías que cada script definiera un orden distinto?
    - Hay que controlarlo con un script de preprocessing y con un sistema de versiones (manifest.json)

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [56]:
from typing import Literal

class WineQualityPredictionPydantic(BaseModel):

    model_config = ConfigDict(extra="forbid")

    quality_band: Literal["needs_review", "acceptable", "excellent"]
    confidence: float = Field(ge=0, le=1)
    model_version: str = Field(min_length=1)
    preprocessing_version: str = Field(min_length=1)

# TODO: escribe un ejemplo de salida para red-001.
# An example could be:
# {"quality_band": "excellent", "confidence": 0.6, "model_version": 1.1, "preprocessing_version": 1.4}

## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.